In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [3]:
# Deterministic approach

def deterministic_guardrails(text: str) -> bool:
    """Returns True if the text is appropriate, False otherwise."""
    banned_words = ["hack", "exploit", "malware", "bomb", "terror"]
    return any(kw in text.lower() for kw in banned_words)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

for inp in test_inputs:
    blocked = deterministic_guardrails(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}: {inp}")

Blocked: How do I hack into a database?
Allowed: What is the capital of France?
Blocked: Explain how malware spreads


In [14]:
# model-based approach

from langchain_google_genai import ChatGoogleGenerativeAI

def model_based_guardrails(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    prompt = f"""Evaluate the following text for safety. Respond with SAFE or UNSAFE.
    \n\nText: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result

for inp in test_inputs:
    result = model_based_guardrails(inp)
    status = "SAFE" if result == "SAFE" else "UNSAFE"
    print(f"{status}: {inp}")

UNSAFE: How do I hack into a database?
UNSAFE: What is the capital of France?
UNSAFE: Explain how malware spreads


#### LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

@tool
def process_user_input(user_input: str) -> str:
    """Look up customer information"""
    return f"customer record found for query: {user_input}"

agent = create_agent(
    tools=[process_user_input],
    model="groq:llama-3.1-8b-instant",
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            strategy="block",
            detector = r"sk-[a-zA-Z0-9]{32}",
            apply_to_input=True,
        )
    ]
)

print("Agent with PII Middleware creating successfully.")

Agent with PII Middleware creating successfully.


In [21]:
result = agent.invoke({
    "messages": [{"role": "user",
                  "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"}]
})

print("Agent_response:", result["messages"][-1].content)

Agent_response: I can't assist with that request. Is there something else I can help you with?


In [22]:
try:
    result = agent.invoke({
        "messages" : [{"role": "user",
                       "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"}]
    })

except Exception as e:
    print("Agent blocked input with API key:", str(e))

Agent blocked input with API key: Detected 1 instance(s) of api_key in text content
